In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import StandardScaler
#一、下载 NLTK 资源
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
#二、读取清洗后数据
df = pd.read_csv(
    "ADHD_清洗后数据.csv"
)
print("读取数据量：", len(df))
#三、文本预处理
#英文停用词
stop_words = set(
    stopwords.words("english")
)
#词形还原器
lemmatizer = WordNetLemmatizer()
#文本预处理函数
def preprocess_text(text):
    # 空值处理
    if pd.isna(text):
        return ""
    # 转字符串
    text = str(text)
    # 分词
    words = word_tokenize(text)
    processed_words = []
    for word in words:
        # 只保留字母
        if word.isalpha():
            # 转小写
            word = word.lower()
            # 去停用词
            if word not in stop_words:
                # 词形还原
                word = lemmatizer.lemmatize(
                    word
                )
                processed_words.append(
                    word
                )
    return " ".join(processed_words)
#处理标题
print("\n开始处理标题文本...")
df["title_processed"] = df[
    "title_clean"
].apply(preprocess_text)
#处理摘要
print("开始处理摘要文本...")
df["abstract_processed"] = df[
    "abstract_clean"
].apply(preprocess_text)
print("\n文本预处理完成")
#四、数值预处理
numeric_cols = [
    "publication_year",
    "cited_by_count",
    "authors_count",
    "institutions_count",
    "countries_count",
    "concepts_count",
    "topics_count",
    "title_length",
    "abstract_length",
    "referenced_works_count",
    "recent_citations",
    "citation_percentile"
]
#数值类型转换
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )
#缺失值填充
# 使用中位数
for col in numeric_cols:
    if col in df.columns:
        median_value = df[col].median()
        df[col] = df[col].fillna(
            median_value
        )
print("\n数值缺失值处理完成")
#五、布尔变量编码
bool_cols = [
    "is_oa",
    "international_collab",
    "multi_institution",
    "is_highly_cited"
]
for col in bool_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .map({
                "true": 1,
                "false": 0,
                "1": 1,
                "0": 0
            })
        )
        # 防止空值
        df[col] = df[col].fillna(0)
print("\n布尔变量编码完成")
#六、数值标准化
#需要标准化的字段
scale_cols = [
    "authors_count",
    "institutions_count",
    "countries_count",
    "concepts_count",
    "topics_count",
    "title_length",
    "abstract_length",
    "referenced_works_count",
    "recent_citations"
]
#创建标准化数据副本
df_scaled = df.copy()
#标准化器
scaler = StandardScaler()
#标准化
df_scaled[scale_cols] = (
    scaler.fit_transform(
        df[scale_cols]
    )
)
print("\n数值标准化完成")
#七、数据检查
print("\n预处理后数据维度：")
print(df.shape)
print("\n缺失值总数：")
print(df.isnull().sum().sum())
print("\n标准化字段：")
print(scale_cols)
#八、保存结果
#预处理数据（未标准化）
# 用于文本分析
df.to_csv(
    "ADHD_预处理后数据.csv",
    index=False,
    encoding="utf-8-sig"
)
#标准化数据
df_scaled.to_csv(
    "ADHD_标准化数据.csv",
    index=False,
    encoding="utf-8-sig"
)
#九、输出结果
print("\n数据预处理完成！")
print("\n文件已保存：")
print("1. ADHD_预处理后数据.csv")
print("2. ADHD_标准化数据.csv")

C:\Users\21116\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\21116\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\21116\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\21116\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


读取数据量： 6551

开始处理标题文本...
开始处理摘要文本...

文本预处理完成

数值缺失值处理完成

布尔变量编码完成

数值标准化完成

预处理后数据维度：
(6551, 30)

缺失值总数：
2254

标准化字段：
['authors_count', 'institutions_count', 'countries_count', 'concepts_count', 'topics_count', 'title_length', 'abstract_length', 'referenced_works_count', 'recent_citations']

数据预处理完成！

文件已保存：
1. ADHD_预处理后数据.csv
2. ADHD_标准化数据.csv
